# Layer 1 Validation Experiments

Validates four assumptions before investing in Layer 2 model improvements.
See `docs/plans/2026-03-09-layer1-validation-design.md` for the full design.

## Setup

In [ ]:
import sys
from pathlib import Path

# Navigate to model/ root so all relative paths work consistently
_nb_dir = Path.cwd()
_model_dir = _nb_dir
while _model_dir.name != 'model' and _model_dir != _model_dir.parent:
    _model_dir = _model_dir.parent
if _model_dir.name == 'model':
    get_ipython().run_line_magic('cd', str(_model_dir))

sys.path.insert(0, 'src') if 'src' not in sys.path else None

import json
import logging

import jsonlines
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import pytorch_lightning as pl

pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision("medium")
logging.basicConfig(level=logging.INFO)

from model_improvement.audio_encoders import MuQLoRAModel
from model_improvement.taxonomy import DIMENSIONS, load_composite_labels
from model_improvement.layer1_validation import (
    score_competition_segments,
    competition_correlation,
    dynamic_range_analysis,
)

DATA_DIR = Path("data")
CHECKPOINT_DIR = DATA_DIR / "checkpoints/model_improvement"
COMPETITION_DIR = DATA_DIR / "competition_cache/chopin2021"
PERCEPIANO_DIR = DATA_DIR / "percepiano_cache"

print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR.resolve()}")

## Experiment 1: Competition Correlation

**Question:** Does A1's quality signal predict expert competition placement?

**Data:** 2,293 segments from Chopin 2021 (already on GDrive, synced locally).

**Decision gate:** rho > 0.3 on at least one aggregation = model signal is real.

In [4]:
# Load A1 fold 3 checkpoint
a1_ckpt = sorted(CHECKPOINT_DIR.glob("A1/fold_3/*.ckpt"))[0]
print(f"Loading A1 from {a1_ckpt.name}")

a1_model = MuQLoRAModel.load_from_checkpoint(
    str(a1_ckpt),
    use_pretrained_muq=False,
)
a1_model.eval()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
a1_model = a1_model.to(device)

# Load competition embeddings
emb_dir = COMPETITION_DIR / "muq_embeddings"
embeddings = {}
for pt_file in sorted(emb_dir.glob("*.pt")):
    embeddings[pt_file.stem] = torch.load(pt_file, map_location="cpu", weights_only=True)
print(f"Loaded {len(embeddings)} competition segment embeddings")

# Load metadata
with jsonlines.open(COMPETITION_DIR / "metadata.jsonl") as reader:
    metadata = list(reader)
print(f"Loaded {len(metadata)} metadata records")
print(f"Performers: {len(set(m['performer'] for m in metadata))}")
print(f"Rounds: {sorted(set(m['round'] for m in metadata))}")

IndexError: list index out of range

In [ ]:
# Score all segments
segment_scores = score_competition_segments(a1_model, embeddings)
print(f"Scored {len(segment_scores)} segments")

# Compute correlations
corr_results = competition_correlation(segment_scores, metadata)

# Display results
print("\n=== Competition Correlation Results ===\n")
for agg_name, corr in corr_results.items():
    gate = "PASS" if abs(corr["rho"]) > 0.3 else "INVESTIGATE" if abs(corr["rho"]) > 0.2 else "FAIL"
    print(f"{agg_name:>8s}: rho={corr['rho']:+.3f}  p={corr['p_value']:.4f}  n={corr['n_performers']}  [{gate}]")
    for dim_name, dim_corr in corr["per_dimension"].items():
        print(f"           {dim_name:>14s}: rho={dim_corr['rho']:+.3f}  p={dim_corr['p_value']:.4f}")
    print()

In [ ]:
# Scatter plots: per-aggregation, overall score vs placement
from collections import defaultdict

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, agg_name in zip(axes, ["mean", "median", "min"]):
    perf_segs = defaultdict(list)
    perf_place = {}
    for m in metadata:
        if m["segment_id"] in segment_scores:
            perf_segs[m["performer"]].append(segment_scores[m["segment_id"]])
            perf_place[m["performer"]] = m["placement"]

    agg_fn = {"mean": np.mean, "median": np.median, "min": np.min}[agg_name]
    performers = sorted(perf_segs.keys())
    agg_overall = [agg_fn(perf_segs[p], axis=0).mean() for p in performers]
    placements = [perf_place[p] for p in performers]

    ax.scatter(placements, agg_overall, alpha=0.7)
    ax.set_xlabel("Competition Placement (lower = better)")
    ax.set_ylabel(f"A1 Score ({agg_name})")
    ax.set_title(f"{agg_name}: rho={corr_results[agg_name]['rho']:+.3f}")
    ax.invert_xaxis()

plt.tight_layout()
plt.savefig(COMPETITION_DIR / "correlation_scatter.png", dpi=150)
plt.show()

In [ ]:
# Per-dimension correlation heatmap
fig, ax = plt.subplots(figsize=(10, 4))
data = []
for agg_name in ["mean", "median", "min"]:
    row = [corr_results[agg_name]["per_dimension"][d]["rho"] for d in DIMENSIONS]
    data.append(row)

sns.heatmap(
    np.array(data), annot=True, fmt=".2f", cmap="RdYlGn",
    xticklabels=DIMENSIONS, yticklabels=["mean", "median", "min"],
    ax=ax, center=0, vmin=-0.6, vmax=0.6,
)
ax.set_title("Per-Dimension Spearman rho vs Competition Placement")
plt.tight_layout()
plt.savefig(COMPETITION_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

## Experiment 3: Dynamic Range at Intermediate Level

**Question:** Does A1 produce meaningful score variance on intermediate-level recordings?

**Data:** YouTube recordings of intermediate pianists.
Must be collected first using yt-dlp (manual search for student recitals).

**No hard gate.** Diagnostic only -- informs Layer 3 data priorities.

In [ ]:
# This cell handles downloading and embedding intermediate recordings.
# Run once, then skip on subsequent notebook runs.

from model_improvement.audio_utils import load_audio, segment_audio
from audio_experiments.extractors.muq import MuQExtractor

INTERMEDIATE_DIR = DATA_DIR / "intermediate_cache"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_AUDIO = INTERMEDIATE_DIR / "audio"
INTERMEDIATE_AUDIO.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_EMB = INTERMEDIATE_DIR / "muq_embeddings"
INTERMEDIATE_EMB.mkdir(parents=True, exist_ok=True)

# YouTube URLs for intermediate performances
# Manually curated: student recitals, diploma exams, amateur performances
# of pieces in PercePiano/MAESTRO repertoire.
# Format: (url, performer_label, piece_label)
INTERMEDIATE_URLS = [
    # TODO: Fill with actual URLs after manual YouTube search
]

# Download, segment, and extract embeddings (skip if already done)
if INTERMEDIATE_URLS and not list(INTERMEDIATE_EMB.glob("*.pt")):
    import subprocess

    for url, performer, piece in INTERMEDIATE_URLS:
        out_path = INTERMEDIATE_AUDIO / f"{performer}_{piece}.wav"
        if out_path.exists():
            continue
        subprocess.run([
            "yt-dlp", "--extract-audio", "--audio-format", "wav",
            "--postprocessor-args", "ffmpeg:-ar 24000 -ac 1",
            "--output", str(out_path.with_suffix(".%(ext)s")),
            "--no-playlist", "--quiet", url,
        ], timeout=300)

    extractor = MuQExtractor(cache_dir=INTERMEDIATE_EMB)
    for wav_file in sorted(INTERMEDIATE_AUDIO.glob("*.wav")):
        audio, sr = load_audio(wav_file, target_sr=24000)
        segments = segment_audio(audio, sr=sr, segment_duration=30.0)
        for i, seg in enumerate(segments):
            seg_id = f"{wav_file.stem}_seg{i:03d}"
            if (INTERMEDIATE_EMB / f"{seg_id}.pt").exists():
                continue
            audio_tensor = torch.from_numpy(seg["audio"]).float()
            embedding = extractor.extract_from_audio(audio_tensor)
            torch.save(embedding, INTERMEDIATE_EMB / f"{seg_id}.pt")
    del extractor
    print(f"Extracted {len(list(INTERMEDIATE_EMB.glob('*.pt')))} intermediate embeddings")
else:
    print(f"Intermediate embeddings: {len(list(INTERMEDIATE_EMB.glob('*.pt')))} files")

In [ ]:
# Score intermediate segments with A1
int_embeddings = {}
for pt_file in sorted(INTERMEDIATE_EMB.glob("*.pt")):
    int_embeddings[pt_file.stem] = torch.load(pt_file, map_location="cpu", weights_only=True)

if int_embeddings:
    int_scores = score_competition_segments(a1_model, int_embeddings)
else:
    int_scores = {}
    print("No intermediate data yet. Populate INTERMEDIATE_URLS and re-run the download cell.")

# Load PercePiano scores for comparison (advanced reference)
labels = load_composite_labels(DATA_DIR / "composite_labels/composite_labels.json")
percepiano_scores = {k: np.array(v[:6]) for k, v in labels.items()}

# Compare distributions
if int_scores:
    groups = {
        "intermediate": int_scores,
        "advanced (PercePiano)": percepiano_scores,
    }
    groups["professional (Chopin 2021)"] = segment_scores

    range_results = dynamic_range_analysis(groups)

    print("\n=== Dynamic Range Analysis ===")
    for group, stats in range_results["group_stats"].items():
        print(f"{group}: mean={stats['overall_mean']:.3f}, std={stats['overall_std']:.3f}, n={stats['n_segments']}")
    print()
    for key, sep in range_results["separation"].items():
        if isinstance(sep, dict):
            print(f"  {key}: diff={sep['mean_diff']:+.3f}, Cohen's d={sep['cohens_d']:.2f}")

In [ ]:
if int_scores:
    import pandas as pd

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for d, (ax, dim_name) in enumerate(zip(axes, DIMENSIONS)):
        plot_data = []
        group_labels = []
        for group_name, scores_dict in groups.items():
            vals = [s[d] for s in scores_dict.values()]
            plot_data.extend(vals)
            group_labels.extend([group_name] * len(vals))

        df = pd.DataFrame({"Score": plot_data, "Group": group_labels})
        sns.boxplot(data=df, x="Group", y="Score", ax=ax)
        ax.set_title(dim_name)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

    plt.suptitle("Per-Dimension Score Distributions by Skill Level")
    plt.tight_layout()
    plt.savefig(INTERMEDIATE_DIR / "dynamic_range_boxplots.png", dpi=150)
    plt.show()
else:
    print("Skipping box plots -- no intermediate data available yet.")

## Experiment 2: AMT Degradation Test

**Question:** How much does S2's pairwise accuracy drop with transcribed MIDI vs ground-truth?

**Data:** 50 MAESTRO recordings with ground-truth MIDI. Transcribed with YourMT3+ (ceiling) and ByteDance (floor).

**Decision gate:** Per-dimension pairwise drop < 10% = symbolic path viable.

**Note:** This experiment requires installing YourMT3+ and ByteDance Piano Transcription.
AMT transcription takes ~4-8 hours on M4. Run the transcription cells once,
then the assessment cells reuse cached transcriptions.

In [ ]:
with open(DATA_DIR / "maestro_cache/contrastive_mapping.json") as f:
    contrastive_mapping = json.load(f)

from model_improvement.layer1_validation import select_maestro_subset
selected_keys = select_maestro_subset(contrastive_mapping, n_recordings=50)
print(f"Selected {len(selected_keys)} recordings from {len(contrastive_mapping)} pieces")
print(f"Sample keys: {selected_keys[:3]}")

In [ ]:
# AMT transcription -- run once, results are cached.
# Requires: uv pip install piano-transcription-inference
# YourMT3+ must be installed from GitHub: pip install git+https://github.com/mimbres/YourMT3.git
#
# This cell is expensive (~4-8 hours on M4). Skip if transcriptions already exist.

AMT_DIR = DATA_DIR / "amt_cache"
AMT_DIR.mkdir(parents=True, exist_ok=True)
YOURMT3_DIR = AMT_DIR / "yourmt3"
YOURMT3_DIR.mkdir(parents=True, exist_ok=True)
BYTEDANCE_DIR = AMT_DIR / "bytedance"
BYTEDANCE_DIR.mkdir(parents=True, exist_ok=True)

MAESTRO_AUDIO_DIR = DATA_DIR / "maestro_cache/audio"

if not MAESTRO_AUDIO_DIR.exists():
    print("MAESTRO audio not found. Download the 50 selected recordings first:")
    print("  See maestro-v3.0.0.json for download URLs")
    print("  Place WAV/FLAC files in data/maestro_cache/audio/")
else:
    # TODO: Run YourMT3+ transcription
    # TODO: Run ByteDance transcription
    # TODO: Save MIDI outputs to YOURMT3_DIR and BYTEDANCE_DIR
    print("AMT transcription code to be implemented when audio is available")

In [ ]:
# Load S2 model
from model_improvement.symbolic_encoders import GraphNeuralNetworkEncoder
from model_improvement.graph import midi_to_graph
from model_improvement.layer1_validation import amt_degradation_comparison

s2_ckpt = sorted(CHECKPOINT_DIR.glob("S2/fold_3/*.ckpt"))[0]
print(f"Loading S2 from {s2_ckpt.name}")

s2_model = GraphNeuralNetworkEncoder.load_from_checkpoint(str(s2_ckpt))
s2_model.eval()
s2_model = s2_model.to(device)

# Build graphs from ground-truth, YourMT3+, and ByteDance MIDI
# Then run pairwise assessment for each source
# Compare per-dimension accuracy drops
# (Detailed implementation depends on AMT output format)

print("S2 assessment on AMT MIDI: awaiting transcription output")